### Imports & Setup

We import the necessary libraries for data manipulation, NLP embedding generation, machine learning algorithms, and rigorous evaluation metrics from scikit-learn.

In [ ]:
import pandas as pd
import numpy as np
import joblib
import torch
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import LabelEncoder
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, classification_report

### Load Data & Preprocess

We load our dataset and filter out rare diseases. *Academic Note: While the training pipeline filtered for diseases appearing >= 2 times, we must filter for >= 5 times here. Stratified 5-fold Cross-Validation mathematically requires at least 5 samples per class to guarantee 1 sample per fold without crashing.* We then generate the 384-dimensional semantic embeddings using the GPU and encode the target labels.

In [ ]:
# Load dataset
df = pd.read_csv('../dataset/final ayurfit.csv')
df = df.dropna(subset=['Symptoms', 'Disease'])

# Filter to >= 5 to allow for 5-fold Stratified CV
disease_counts = df['Disease'].value_counts()
df = df[df['Disease'].isin(disease_counts[disease_counts >= 5].index)]

# Target Encoding
label_encoder = LabelEncoder()
y = label_encoder.fit_transform(df['Disease'])

# Feature Extraction
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
model = SentenceTransformer('all-MiniLM-L6-v2', device=device)
X = model.encode(df['Symptoms'].tolist(), show_progress_bar=True)

### Test 1 - K-Fold Cross Validation

Cross-Validation ensures the model performs consistently regardless of how the data is split.

In [ ]:
cv_svm_model = LinearSVC(random_state=42, dual=False)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print("Running 5-Fold Stratified Cross-Validation...")
cv_scores = cross_val_score(cv_svm_model, X, y, cv=skf, scoring='accuracy')

print(f"Accuracy for all 5 folds: {cv_scores}")
print(f"Mean Accuracy: {cv_scores.mean() * 100:.2f}%")
print(f"Standard Deviation: {cv_scores.std():.4f}")

### Test 2 - Holdout Set Evaluation

Evaluating on a strict 80/20 train-test split to generate a detailed classification report.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

holdout_svm_model = LinearSVC(random_state=42, dual=False)
holdout_svm_model.fit(X_train, y_train)
y_pred = holdout_svm_model.predict(X_test)

overall_accuracy = accuracy_score(y_test, y_pred)
print(f"Overall Holdout Accuracy: {overall_accuracy * 100:.2f}%\n")

present_classes = np.unique(np.concatenate((y_test, y_pred)))
target_names_present = label_encoder.inverse_transform(present_classes)
print(classification_report(y_test, y_pred, labels=present_classes, target_names=target_names_present, zero_division=0))

### Load Saved Models

We load the serialized artifacts generated by our training pipeline to ensure the export mechanism was successful and the production models are ready for inference.

In [ ]:
saved_svm_model = joblib.load('tier1_svm_model.joblib')
saved_label_encoder = joblib.load('disease_label_encoder.joblib')
print("Models successfully loaded from disk!")

### Test 3 - Manual Real-World Sanity Checks

Testing the saved model with unseen, raw user inputs to verify real-world prediction capabilities.

In [ ]:
def test_unseen_symptoms(text):
    embedding = model.encode([text], show_progress_bar=False)
    predicted_encoded = saved_svm_model.predict(embedding)
    predicted_disease = saved_label_encoder.inverse_transform(predicted_encoded)[0]
    print(f"Input: '{text}'")
    print(f"Prediction: {predicted_disease}\n")

test_cases = [
    "I have a severe headache, my vision is blurry, and I feel nauseous.",
    "My chest hurts really bad when I breathe in, and I have a wet cough.",
    "I am constantly thirsty, peeing all the time, and feeling very tired.",
    "My joints are stiff and swollen, especially in the mornings.",
    "I have a red, itchy rash all over my arms that won't go away."
]

for case in test_cases:
    test_unseen_symptoms(case)